# Match Pattern To Master Sphere

## 1. 输入实验菊池图并校正到球面

这一步保留整张 Kikuchi 图在球面上的连续 patch，用来显示和导出。匹配时再从里面抽样亮点，提高速度。

In [ ]:
from pathlib import Path
import json
import subprocess
import webbrowser

import h5py
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import vtk
from IPython.display import HTML, IFrame, display
from matplotlib import cm
from scipy import ndimage as ndi
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial.transform import Rotation as R
from skimage import exposure, filters, morphology

# =========================
# Input: edit these values first
# pattern_path: experimental EBSD pattern image path
# pcx, pcy, pcz: pattern center parameters
# =========================
pattern_path = Path("patterns/1.png")
pcx, pcy, pcz = 0.5, 0.5, 0.6

master_h5_path = Path(".venv/Lib/site-packages/kikuchipy/data/emsoft_ebsd_master_pattern/ni_mc_mp_20kv_uint8_gzip_opts9.h5")
output_dir = Path("outputs")

mask_erosion = 6
background_sigma = 9.0
band_sigma_min = 1
band_sigma_max = 5
match_quantile = 0.82
top_k_points = 12000
score_line_weight = 0.75
score_intensity_weight = 0.25

coarse_rotation_count = 1000
refine_schedule = [(8, 300), (3, 500), (1, 500)]
random_seed = 0

sphere_lon_count = 180
sphere_colat_count = 90

detector_conventions = {
    "identity": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "flip_x": np.array([[-1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "flip_y": np.array([[1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "flip_xy": np.array([[-1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "swap_xy": np.array([[0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "swap_xy_flip_x": np.array([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "swap_xy_flip_y": np.array([[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
    "swap_xy_flip_xy": np.array([[0.0, -1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32),
}

auto_open_native_preview = True
show_html_preview_in_notebook = False
auto_open_browser_preview = False
auto_open_vscode_preview = False
vscode_open_delay_ms = 1600
preview_patch_radius_scale = 1.0
native_preview_window_size = (1100, 850)

print("Input pattern path:", pattern_path)
print(f"Input pattern center: pcx={pcx:.4f}, pcy={pcy:.4f}, pcz={pcz:.4f}")


In [ ]:
def zscore01(values, mask):
    output = np.zeros_like(values, dtype=np.float32)
    masked_values = values[mask]
    normalized = (masked_values - masked_values.mean()) / (masked_values.std() + 1e-8)
    normalized = np.clip(normalized, -2.5, 2.5)
    normalized = (normalized + 2.5) / 5.0
    output[mask] = normalized.astype(np.float32)
    return output

def preprocess_pattern(gray_image, valid_mask):
    gray_image = exposure.rescale_intensity(gray_image.astype(np.float32), in_range="image", out_range=(0.0, 1.0))
    valid_mask = ndi.binary_erosion(valid_mask, iterations=mask_erosion)
    valid_mask = morphology.remove_small_holes(valid_mask, area_threshold=64)
    valid_mask = morphology.remove_small_objects(valid_mask, min_size=256)

    background = filters.gaussian(gray_image, sigma=background_sigma)
    corrected = gray_image - background
    corrected[~valid_mask] = 0.0
    corrected = exposure.rescale_intensity(corrected, in_range="image", out_range=(0.0, 1.0))

    band_response = filters.meijering(
        corrected,
        sigmas=range(band_sigma_min, band_sigma_max + 1),
        black_ridges=False,
    )
    band_response = exposure.rescale_intensity(band_response, in_range="image", out_range=(0.0, 1.0))
    band_response[~valid_mask] = 0.0

    corrected_score = zscore01(corrected, valid_mask)
    band_score = zscore01(band_response, valid_mask)
    combined_response = score_intensity_weight * corrected_score + score_line_weight * band_score
    combined_response[~valid_mask] = 0.0
    return gray_image, background, corrected, band_response, corrected_score, band_score, combined_response, valid_mask

def build_patch_mesh(points_grid, rgba_grid, valid_mask):
    valid_indices = np.flatnonzero(valid_mask.ravel())
    vertex_lookup = -np.ones(valid_mask.size, dtype=np.int32)
    vertex_lookup[valid_indices] = np.arange(len(valid_indices), dtype=np.int32)

    faces = []
    for r in range(valid_mask.shape[0] - 1):
        for c in range(valid_mask.shape[1] - 1):
            if valid_mask[r, c] and valid_mask[r + 1, c] and valid_mask[r + 1, c + 1] and valid_mask[r, c + 1]:
                p00 = int(vertex_lookup[r * valid_mask.shape[1] + c])
                p10 = int(vertex_lookup[(r + 1) * valid_mask.shape[1] + c])
                p11 = int(vertex_lookup[(r + 1) * valid_mask.shape[1] + c + 1])
                p01 = int(vertex_lookup[r * valid_mask.shape[1] + c + 1])
                faces.extend([3, p00, p10, p11, 3, p00, p11, p01])

    mesh = pv.PolyData(points_grid.reshape(-1, 3)[valid_indices], np.array(faces, dtype=np.int32))
    mesh["rgba"] = rgba_grid.reshape(-1, 4)[valid_indices]
    return mesh

img = imageio.imread(pattern_path)
if img.ndim == 3:
    exp_image_raw = img[..., :3].mean(axis=2).astype(np.float32)
    detector_mask_raw = img[..., 3] > 0 if img.shape[2] == 4 else np.ones(img.shape[:2], dtype=bool)
else:
    exp_image_raw = img.astype(np.float32)
    detector_mask_raw = exp_image_raw > 0

exp_image, exp_background, exp_corrected, exp_band, exp_corrected_score, exp_band_score, exp_combined_response, detector_mask = preprocess_pattern(exp_image_raw, detector_mask_raw)
H, W = exp_image.shape

jj, ii = np.meshgrid(np.arange(W), np.arange(H))
cx = pcx * (W - 1)
cy = pcy * (H - 1)

X = (jj - cx) / (pcz * H)
Y = -(ii - cy) / (pcz * H)
Z = np.ones_like(X)
norm = np.sqrt(X**2 + Y**2 + Z**2) + 1e-8
X /= norm
Y /= norm
Z /= norm

full_points_grid = np.stack([X, Y, Z], axis=-1)
full_points = full_points_grid.reshape(-1, 3)

patch_rgba = cm.gray(exp_corrected)
patch_rgba[..., 3] = detector_mask.astype(np.float32)
patch_rgba_u8 = (patch_rgba * 255).astype(np.uint8)

match_threshold = np.quantile(exp_combined_response[detector_mask], match_quantile)
match_mask = detector_mask & (exp_combined_response >= match_threshold)
exp_points = np.column_stack([X[match_mask], Y[match_mask], Z[match_mask]])
exp_band_values = exp_band[match_mask]
exp_intensity_values = exp_corrected[match_mask]
exp_combined_values = exp_combined_response[match_mask]

if len(exp_combined_values) > top_k_points:
    keep = np.argsort(exp_combined_values)[-top_k_points:]
    exp_points = exp_points[keep]
    exp_band_values = exp_band_values[keep]
    exp_intensity_values = exp_intensity_values[keep]
    exp_combined_values = exp_combined_values[keep]

exp_band_values_z = (exp_band_values - exp_band_values.mean()) / (exp_band_values.std() + 1e-8)
exp_intensity_values_z = (exp_intensity_values - exp_intensity_values.mean()) / (exp_intensity_values.std() + 1e-8)
patch_mesh = build_patch_mesh(full_points_grid, patch_rgba_u8, detector_mask)

print(f"Pattern: {pattern_path}")
print(f"Image shape: {H} x {W}")
print(f"Continuous patch vertices: {detector_mask.sum()}")
print(f"Combined response threshold: {match_threshold:.4f}")
print(f"Sampled points for matching: {len(exp_points)}")

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes[0, 0].imshow(exp_image, cmap="gray")
axes[0, 0].set_title("Original Pattern")
axes[0, 0].axis("off")

axes[0, 1].imshow(exp_background, cmap="gray")
axes[0, 1].set_title("Step 1: Background")
axes[0, 1].axis("off")

axes[0, 2].imshow(exp_corrected, cmap="gray")
axes[0, 2].contour(detector_mask.astype(float), levels=[0.5], colors=["cyan"], linewidths=0.7)
axes[0, 2].set_title("Step 1: Corrected")
axes[0, 2].axis("off")

axes[1, 0].imshow(exp_band, cmap="inferno")
axes[1, 0].set_title("Step 2: Ridge Enhanced")
axes[1, 0].axis("off")

axes[1, 1].imshow(exp_combined_response, cmap="viridis")
axes[1, 1].set_title("Step 3: Combined Response")
axes[1, 1].axis("off")

axes[1, 2].imshow(exp_corrected, cmap="gray")
axes[1, 2].imshow(match_mask.astype(float), cmap="autumn", alpha=0.55)
axes[1, 2].set_title("Step 3: Matching Samples")
axes[1, 2].axis("off")

preprocess_png_path = output_dir / f"{pattern_path.stem}_preprocess_pipeline.png"
output_dir.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(preprocess_png_path, dpi=200, bbox_inches="tight")
print(f"Saved preprocessing figure: {preprocess_png_path}")
plt.show()

fig = plt.figure(figsize=(10, 4.5))
ax1 = fig.add_subplot(121, projection="3d")
ax1.plot_surface(
    X,
    Y,
    Z,
    facecolors=patch_rgba,
    rstride=1,
    cstride=1,
    linewidth=0,
    antialiased=False,
    shade=False,
)
ax1.set_title("Continuous Patch On Unit Sphere")
ax1.set_box_aspect((1, 1, 1))
ax1.set_xlim(-1, 1)
ax1.set_ylim(-1, 1)
ax1.set_zlim(-1, 1)
ax1.axis("off")

ax2 = fig.add_subplot(122)
ax2.imshow(exp_combined_response, cmap="viridis")
ax2.set_title("Combined Response Used For Sampling")
ax2.axis("off")
plt.tight_layout()
plt.show()


## 2. 读取标准 Kikuchi sphere，并与校正后的实验 patch 做匹配

In [ ]:
with h5py.File(master_h5_path, "r") as f:
    master_upper = f["EMData/EBSDmaster/masterSPNH"][0].astype(np.float32) / 255.0
    master_lower = f["EMData/EBSDmaster/masterSPSH"][0].astype(np.float32) / 255.0

master_upper_blur = filters.gaussian(master_upper, sigma=background_sigma)
master_lower_blur = filters.gaussian(master_lower, sigma=background_sigma)
master_upper_corrected = exposure.rescale_intensity(master_upper - master_upper_blur, in_range="image", out_range=(0.0, 1.0))
master_lower_corrected = exposure.rescale_intensity(master_lower - master_lower_blur, in_range="image", out_range=(0.0, 1.0))
master_upper_band = exposure.rescale_intensity(
    filters.meijering(master_upper_corrected, sigmas=range(band_sigma_min, band_sigma_max + 1), black_ridges=False),
    in_range="image",
    out_range=(0.0, 1.0),
)
master_lower_band = exposure.rescale_intensity(
    filters.meijering(master_lower_corrected, sigmas=range(band_sigma_min, band_sigma_max + 1), black_ridges=False),
    in_range="image",
    out_range=(0.0, 1.0),
)

master_axis = np.linspace(-1.0, 1.0, master_upper.shape[0])
upper_interp = RegularGridInterpolator((master_axis, master_axis), master_upper_corrected, bounds_error=False, fill_value=0.0)
lower_interp = RegularGridInterpolator((master_axis, master_axis), master_lower_corrected, bounds_error=False, fill_value=0.0)
upper_band_interp = RegularGridInterpolator((master_axis, master_axis), master_upper_band, bounds_error=False, fill_value=0.0)
lower_band_interp = RegularGridInterpolator((master_axis, master_axis), master_lower_band, bounds_error=False, fill_value=0.0)

def sample_master(vectors, upper_sampler=upper_interp, lower_sampler=lower_interp):
    x = vectors[:, 0]
    y = vectors[:, 1]
    z = vectors[:, 2]
    sampled = np.zeros(len(vectors), dtype=np.float32)

    upper = z >= 0
    lower = ~upper

    if np.any(upper):
        xy_upper = np.column_stack([
            y[upper] / (1.0 + z[upper] + 1e-8),
            x[upper] / (1.0 + z[upper] + 1e-8),
        ])
        sampled[upper] = upper_sampler(xy_upper)

    if np.any(lower):
        xy_lower = np.column_stack([
            y[lower] / (1.0 - z[lower] + 1e-8),
            x[lower] / (1.0 - z[lower] + 1e-8),
        ])
        sampled[lower] = lower_sampler(xy_lower)

    return sampled

def score_rotation(rotation):
    rotated = rotation.apply(exp_points)
    master_band_values = sample_master(rotated, upper_band_interp, lower_band_interp)
    master_intensity_values = sample_master(rotated, upper_interp, lower_interp)
    master_band_values_z = (master_band_values - master_band_values.mean()) / (master_band_values.std() + 1e-8)
    master_intensity_values_z = (master_intensity_values - master_intensity_values.mean()) / (master_intensity_values.std() + 1e-8)
    line_score = float(np.mean(exp_band_values_z * master_band_values_z))
    intensity_score = float(np.mean(exp_intensity_values_z * master_intensity_values_z))
    return score_line_weight * line_score + score_intensity_weight * intensity_score

base_exp_points = exp_points.copy()
base_full_points = full_points.copy()

rng = np.random.default_rng(random_seed)
best_rotation = None
best_score = -np.inf
best_detector_convention = None
best_detector_transform = None

for convention_name, detector_transform in detector_conventions.items():
    exp_points = base_exp_points @ detector_transform.T
    coarse_rotations = R.random(coarse_rotation_count, random_state=rng)
    coarse_scores = np.array([score_rotation(rot) for rot in coarse_rotations])
    convention_rotation = coarse_rotations[int(np.argmax(coarse_scores))]
    convention_score = float(coarse_scores.max())

    for step_deg, attempts in refine_schedule:
        for _ in range(attempts):
            delta = R.from_euler("zyx", rng.normal(scale=step_deg, size=3), degrees=True)
            candidate = delta * convention_rotation
            candidate_score = score_rotation(candidate)
            if candidate_score > convention_score:
                convention_rotation = candidate
                convention_score = candidate_score

    if convention_score > best_score:
        best_score = convention_score
        best_rotation = convention_rotation
        best_detector_convention = convention_name
        best_detector_transform = detector_transform

exp_points = base_exp_points @ best_detector_transform.T
convention_aligned_full_points = base_full_points @ best_detector_transform.T
convention_aligned_full_points_grid = convention_aligned_full_points.reshape(H, W, 3)
best_euler_zyx = best_rotation.as_euler("zyx", degrees=True)

matched_full_points_grid = best_rotation.apply(convention_aligned_full_points).reshape(H, W, 3)
matched_valid_points = matched_full_points_grid[detector_mask]
matched_plot_points_grid = matched_full_points_grid.copy()
matched_plot_points_grid[~detector_mask] = np.nan
patch_mesh = build_patch_mesh(convention_aligned_full_points_grid, patch_rgba_u8, detector_mask)
matched_patch_mesh = patch_mesh.copy(deep=True)
matched_patch_mesh.points = best_rotation.apply(matched_patch_mesh.points) * preview_patch_radius_scale

print(f"Master sphere file: {master_h5_path.name}")
print(f"Best detector convention: {best_detector_convention}")
print(f"Best score: {best_score:.4f}")
print(f"Best Euler ZYX (deg): {best_euler_zyx}")


## 3. 把连续的实验 patch 贴到匹配到的 Kikuchi sphere 方位上

In [ ]:
lon = np.linspace(-np.pi, np.pi, sphere_lon_count)
colat = np.linspace(0.0, np.pi, sphere_colat_count)
LON, COLAT = np.meshgrid(lon, colat)

sphere_x = np.sin(COLAT) * np.cos(LON)
sphere_y = np.sin(COLAT) * np.sin(LON)
sphere_z = np.cos(COLAT)
sphere_vectors = np.column_stack([sphere_x.ravel(), sphere_y.ravel(), sphere_z.ravel()])
sphere_texture = sample_master(sphere_vectors).reshape(COLAT.shape)
sphere_colors = cm.gray(sphere_texture)

matched_lon = np.degrees(np.arctan2(matched_valid_points[:, 1], matched_valid_points[:, 0]))
matched_colat = np.degrees(np.arccos(np.clip(matched_valid_points[:, 2], -1.0, 1.0)))

patch_projection_sum = np.zeros_like(sphere_texture)
patch_projection_count = np.zeros_like(sphere_texture)
matched_theta = np.arctan2(matched_valid_points[:, 1], matched_valid_points[:, 0])
matched_phi = np.arccos(np.clip(matched_valid_points[:, 2], -1.0, 1.0))
u = np.clip(np.round((matched_theta + np.pi) / (2 * np.pi) * (sphere_lon_count - 1)).astype(int), 0, sphere_lon_count - 1)
v = np.clip(np.round(matched_phi / np.pi * (sphere_colat_count - 1)).astype(int), 0, sphere_colat_count - 1)
np.add.at(patch_projection_sum, (v, u), exp_corrected[detector_mask])
np.add.at(patch_projection_count, (v, u), 1)
patch_projection = np.zeros_like(sphere_texture)
projected_mask = patch_projection_count > 0
patch_projection[projected_mask] = patch_projection_sum[projected_mask] / patch_projection_count[projected_mask]

fig = plt.figure(figsize=(13, 6))

ax1 = fig.add_subplot(121, projection="3d")
ax1.plot_surface(
    sphere_x,
    sphere_y,
    sphere_z,
    facecolors=sphere_colors,
    rstride=1,
    cstride=1,
    linewidth=0,
    antialiased=False,
    shade=False,
    alpha=0.58,
)
ax1.plot_surface(
    matched_plot_points_grid[:, :, 0],
    matched_plot_points_grid[:, :, 1],
    matched_plot_points_grid[:, :, 2],
    facecolors=patch_rgba,
    rstride=1,
    cstride=1,
    linewidth=0,
    antialiased=False,
    shade=False,
)
ax1.set_title("Continuous Patch Matched Onto Master Sphere")
ax1.set_box_aspect((1, 1, 1))
ax1.set_xlim(-1, 1)
ax1.set_ylim(-1, 1)
ax1.set_zlim(-1, 1)
ax1.axis("off")

ax2 = fig.add_subplot(122)
ax2.imshow(sphere_texture, cmap="gray", origin="upper", extent=[-180, 180, 180, 0], aspect="auto")
ax2.imshow(
    patch_projection,
    cmap="gray",
    origin="upper",
    extent=[-180, 180, 180, 0],
    aspect="auto",
    alpha=np.where(projected_mask, 0.95, 0.0),
)
ax2.set_title("Matched Kikuchi Patch On Spherical Map")
ax2.set_xlabel("Longitude (deg)")
ax2.set_ylabel("Colatitude (deg)")
plt.tight_layout()

output_dir.mkdir(parents=True, exist_ok=True)
matched_png_path = output_dir / f"{pattern_path.stem}_matched_to_master_sphere.png"
plt.savefig(matched_png_path, dpi=200, bbox_inches="tight")
print(f"Saved figure: {matched_png_path}")
plt.show()


## 4. 导出连续 patch 的 glTF 和交互查看器

In [ ]:
sphere_rgba_u8 = (sphere_colors * 255).astype(np.uint8)

sphere_grid = pv.StructuredGrid(sphere_x, sphere_y, sphere_z)
sphere_surface = sphere_grid.extract_surface(algorithm='dataset_surface')
sphere_surface["rgba"] = sphere_rgba_u8.reshape(-1, 4)

plotter = pv.Plotter(off_screen=True)
plotter.set_background("white")
plotter.add_mesh(sphere_surface, scalars="rgba", rgb=True, smooth_shading=False, opacity=1.0, lighting=False)
patch_actor = plotter.add_mesh(matched_patch_mesh, scalars="rgba", rgb=True, smooth_shading=False, lighting=False)
patch_actor.GetMapper().SetResolveCoincidentTopologyToPolygonOffset()
patch_actor.GetMapper().SetRelativeCoincidentTopologyPolygonOffsetParameters(-8.0, -8.0)

rotatable_gltf_path = output_dir / f"{pattern_path.stem}_matched_to_master_sphere.gltf"
plotter.export_gltf(rotatable_gltf_path)
plotter.close()

if auto_open_native_preview:
    try:
        native_plotter = pv.Plotter(window_size=native_preview_window_size)
        native_plotter.set_background("white")
        native_plotter.add_mesh(sphere_surface, scalars="rgba", rgb=True, smooth_shading=False, opacity=1.0, lighting=False)
        native_patch_actor = native_plotter.add_mesh(matched_patch_mesh, scalars="rgba", rgb=True, smooth_shading=False, opacity=1.0, lighting=False)
        native_patch_actor.GetMapper().SetResolveCoincidentTopologyToPolygonOffset()
        native_patch_actor.GetMapper().SetRelativeCoincidentTopologyPolygonOffsetParameters(-8.0, -8.0)
        native_plotter.add_axes()
        native_plotter.camera_position = "xy"
        native_plotter.show(title="Matched corrected Kikuchi patch", interactive=True, auto_close=True)
    except Exception as exc:
        print(f"Native PyVista preview failed: {exc}")

gltf_text = rotatable_gltf_path.read_text(encoding="utf-8")
viewer_html_path = output_dir / f"{pattern_path.stem}_matched_to_master_sphere_viewer.html"

viewer_html = f"""
<!doctype html>
<html>
<head>
  <meta charset=\"utf-8\" />
  <title>Kikuchi Sphere Viewer</title>
  <style>
    html, body {{ margin: 0; padding: 0; background: #ffffff; overflow: hidden; }}
    #wrap {{ width: 100vw; height: 100vh; }}
    canvas {{ display: block; }}
    #label {{ position: absolute; top: 12px; left: 12px; padding: 8px 10px; background: rgba(255,255,255,0.85); font: 12px/1.4 sans-serif; border-radius: 8px; }}
  </style>
</head>
<body>
  <div id=\"wrap\"></div>
  <div id=\"label\">Drag: rotate | Wheel: zoom | Right-drag: pan</div>
  <script type=\"module\">
    import * as THREE from 'https://unpkg.com/three@0.160.0/build/three.module.js';
    import {{ OrbitControls }} from 'https://unpkg.com/three@0.160.0/examples/jsm/controls/OrbitControls.js';
    import {{ GLTFLoader }} from 'https://unpkg.com/three@0.160.0/examples/jsm/loaders/GLTFLoader.js';

    const container = document.getElementById('wrap');
    const scene = new THREE.Scene();
    scene.background = new THREE.Color(0xffffff);

    const camera = new THREE.PerspectiveCamera(45, container.clientWidth / container.clientHeight, 0.01, 100);
    camera.position.set(0, 0, 3.0);

    const renderer = new THREE.WebGLRenderer({{ antialias: true }});
    renderer.setPixelRatio(window.devicePixelRatio);
    renderer.setSize(container.clientWidth, container.clientHeight);
    container.appendChild(renderer.domElement);

    const controls = new OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.dampingFactor = 0.05;

    scene.add(new THREE.AmbientLight(0xffffff, 1.15));
    const dir = new THREE.DirectionalLight(0xffffff, 1.0);
    dir.position.set(3, 3, 4);
    scene.add(dir);

    const gltfText = {json.dumps(gltf_text)};
    const blob = new Blob([gltfText], {{ type: 'model/gltf+json' }});
    const url = URL.createObjectURL(blob);

    const loader = new GLTFLoader();
    loader.load(url, (gltf) => {{
      const meshes = [];
      gltf.scene.traverse((obj) => {{
        if (obj.isMesh) {{
          meshes.push(obj);
          obj.material = obj.material.clone();
          obj.material.vertexColors = true;
          obj.material.side = THREE.DoubleSide;
          obj.material.roughness = 1.0;
        }}
      }});
      if (meshes[0]) {{
        meshes[0].material.transparent = true;
        meshes[0].material.opacity = 0.42;
        meshes[0].material.depthWrite = false;
      }}
      if (meshes[1]) {{
        meshes[1].material.transparent = true;
        meshes[1].material.opacity = 1.0;
        meshes[1].renderOrder = 2;
      }}
      scene.add(gltf.scene);
      const box = new THREE.Box3().setFromObject(gltf.scene);
      const center = box.getCenter(new THREE.Vector3());
      const size = box.getSize(new THREE.Vector3()).length();
      controls.target.copy(center);
      camera.position.set(center.x, center.y, center.z + Math.max(2.2, size * 1.2));
      controls.update();
    }});

    window.addEventListener('resize', () => {{
      camera.aspect = container.clientWidth / container.clientHeight;
      camera.updateProjectionMatrix();
      renderer.setSize(container.clientWidth, container.clientHeight);
    }});

    function animate() {{
      requestAnimationFrame(animate);
      controls.update();
      renderer.render(scene, camera);
    }}
    animate();
  </script>
</body>
</html>
"""

viewer_html_path.write_text(viewer_html, encoding="utf-8")

print(f"Saved rotatable 3D file: {rotatable_gltf_path}")
print(f"Saved interactive viewer: {viewer_html_path}")

if show_html_preview_in_notebook:
    display(HTML(f'<p><a href="{viewer_html_path.as_posix()}" target="_blank">Open interactive viewer in a new tab</a></p>'))
    display(IFrame(src=viewer_html_path.as_posix(), width="100%", height=700))

if auto_open_browser_preview:
    viewer_abs = viewer_html_path.resolve()
    opened = webbrowser.open(viewer_abs.as_uri(), new=2)
    if not opened:
        subprocess.run([
            'powershell',
            '-NoProfile',
            '-ExecutionPolicy', 'Bypass',
            '-Command', f"Start-Process -FilePath '{viewer_abs}'",
        ], check=False)
    print('Opened interactive HTML preview in the default browser.')


if auto_open_vscode_preview:
    gltf_abs = rotatable_gltf_path.resolve()
    ps_script = f'''
    $wshell = New-Object -ComObject WScript.Shell
    Start-Process code.cmd -ArgumentList '-r', '{gltf_abs}'
    Start-Sleep -Milliseconds {vscode_open_delay_ms}
    $null = $wshell.AppActivate('Visual Studio Code')
    Start-Sleep -Milliseconds 300
    $wshell.SendKeys('%g')
    '''
    subprocess.run([
        'powershell',
        '-NoProfile',
        '-ExecutionPolicy', 'Bypass',
        '-Command', ps_script,
    ], check=False)
    print('Tried to open glTF Preview in VS Code automatically.')
